# TransferAttack PGN Generation & Evaluation Pipeline

Make sure your runtime is set to **GPU** (Runtime > Change runtime type -> T4 GPU or better).

In [ ]:
import os
if os.system('nvidia-smi') != 0:
    raise RuntimeError('!!! NO GPU PHYSICALLY ATTACHED !!!\nGo to Runtime > Change runtime type > confirm T4 GPU is selected, then Runtime > Disconnect and delete runtime, then reconnect and re-run from the top.')
print('GPU diagnostic passed.')


In [ ]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU Memory growth enabled.')
    except RuntimeError as e:
        print(e)
else:
    tf.config.set_visible_devices([], 'GPU')
    print('Forcing CPU-only mode.')


### Setup Repository and Requirements

In [ ]:
!git clone -b pgn-attack https://github.com/Chidroopakanaparthy/transferattack.git
%cd transferattack
!pip install -r requirements.txt -q

### Extract Dataset and Setup IR152

This cell downloads the dataset and the PyTorch IR152 weights using gdown.

In [ ]:
!gdown 1CD1NjufQJeImCMjZbI_yDDeWWMmRVvR_
!unzip -q -o dataset_extractedfaces.zip

# Download IR152 models from shared drive folder
!pip install -U gdown
# The drive folder contains ir152.py and ir152.pth
!gdown --folder https://drive.google.com/drive/folders/130GJ-WYNVNwe94pWEB9D-2fm7Uc6TsJl -O core_download/
# ONLY move the .pth file so we don't overwrite the patched ir152.py in the repo
!find core_download/ -name '*.pth' -exec mv {} core/ \;


### Dummy IR152 Test Cell (PyTorch)

In [ ]:
import os
import numpy as np
import tensorflow as tf
import torch
from core.ir152_loader import load_ir152, compute_ir152_embedding

if not os.path.exists('core/ir152.pth'):
    raise FileNotFoundError('core/ir152.pth is MISSING! Download failed.')
else:
    print('IR152 weights found. Attempting to load...')
    ir152_model = load_ir152('core/ir152.pth')
    dummy_img = tf.convert_to_tensor(np.random.uniform(-1, 1, (1, 112, 112, 3)).astype(np.float32))
    emb = compute_ir152_embedding(ir152_model, dummy_img)
    print(f'IR152 successfully loaded! Output shape: {emb.shape}')
    print(f'Sample embeddings min/max: {emb.min():.4f}, {emb.max():.4f}')


### Run Pipeline (Parts A, B, and C)

In [ ]:
# Smoke Test FIRST
!PYTHONPATH=. python scripts/run_pgn_pipeline.py --smoke-test

In [ ]:
# Full Pipeline
!PYTHONPATH=. python scripts/run_pgn_pipeline.py

### Download Results

In [ ]:
from google.colab import files
!zip -r results_pgn.zip results_pgn/
files.download('results_pgn.zip')